# MiniLM compartido + tres MLP independientes · RTX 4070 Ti SUPER · ONNX INT8

Flujo: **ver datos y etiquetas → una cabeza en CUDA → guardar en SSD → cerrar proceso → siguiente cabeza → exportar → validar → medir RAM → inferir**.

Cada tarea tiene su propio DataFrame y un único target. FAQ usa los títulos seleccionados de Stack Exchange como 1 y los comentarios como 0. Logro y sentimiento usan las respuestas de Gemini existentes. Sentimiento: negativo=0, neutro=0.5, mixto=0.5, positivo=1. El indeterminado queda en revisión.

Comparación controlada de cabezas: `384→16→4→1` frente a `384→64→32→16→4→1`. ReLU tras cada capa oculta, Sigmoid al salir. Se comparan sobre los mismos embeddings INT8 y splits.

**Un solo encoder:** MiniLM multilingüe L12, salida 384. Se reutiliza el encoder afinado de la etapa v2 en esta misma RTX. En esta comparación se congela y se entrenan únicamente las cabezas, por separado. La caché se calcula con el encoder INT8 CPU para coincidir con producción. Los tres entrenamientos no duplican ni modifican el encoder. El entrenamiento anterior del encoder está registrado en `artifacts_v2/mlp/training.json`.

El límite de 1,000 MB corresponde al proceso de inferencia. PyTorch de entrenamiento y Jupyter se ejecutan en la PC de 64 GB y no forman parte del servicio OCI. El paquete de producción no importa Torch/pandas/sklearn.


In [1]:
from pathlib import Path
import os,sys,json,gc,subprocess
from IPython.display import display,Code
ROOT=Path.cwd()
if not (ROOT/'train_mlp_head.py').exists():ROOT=ROOT/'2026-10-04'
assert (ROOT/'train_mlp_head.py').exists(),ROOT
OUT=ROOT/'artifacts_mlp_int8'
DEPLOY=ROOT/'deploy_mlp_int8'
TASKS=('es_faq','es_logro','sentimiento')
REENTRENAR=False
RECREAR_CACHE=False
REVALIDAR=False
print('PASO 0 · carpeta:',ROOT)
print('REENTRENAR=',REENTRENAR,'RECREAR_CACHE=',RECREAR_CACHE,'REVALIDAR=',REVALIDAR)

def execute_step(arguments):
    cmd=[sys.executable,'-u',*map(str,arguments)]
    print('LLAMADA EXPLÍCITA:',cmd,flush=True)
    process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace')
    for line in process.stdout:print(line,end='',flush=True)
    status=process.wait()
    if status:raise RuntimeError(f'Paso fallido ({status}); se detiene la secuencia.')
    print('Proceso terminado; sus modelos y memoria quedaron liberados.',flush=True)

def show_source(name):
    print('IMPLEMENTACIÓN:',name)
    display(Code((ROOT/name).read_text(encoding='utf-8-sig'),language='python'))


PASO 0 · carpeta: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04
REENTRENAR= False RECREAR_CACHE= False REVALIDAR= False


## 1. Encontrar y comprobar la RTX, CPU y RAM
Se llama a `nvidia-smi` y luego se comprueba CUDA con PyTorch **en otro proceso**. No se permite continuar un entrenamiento con CPU de forma silenciosa. El script de cada cabeza vuelve a comprobar GPU y muestra el dispositivo de modelo, embeddings y target.


In [2]:
import psutil
gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],capture_output=True,text=True,check=True).stdout.strip()
print('RTX ENCONTRADA:',gpu)
assert '4070 Ti SUPER' in gpu,gpu
if os.name=='nt':
    import winreg
    with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE,r'HARDWARE\DESCRIPTION\System\CentralProcessor\0') as key:
        print('CPU:',winreg.QueryValueEx(key,'ProcessorNameString')[0])
print('Hilos lógicos:',psutil.cpu_count(),'RAM instalada GiB:',round(psutil.virtual_memory().total/2**30,2))
cuda_code="import torch; print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda); assert torch.cuda.is_available(); gpu=torch.cuda.get_device_name(0); print('GPU CUDA:',gpu); assert '4070 Ti SUPER' in gpu; x=torch.ones((1,384),device='cuda'); print('Tensor real:',x.shape,x.device); print('VRAM GiB:',torch.cuda.get_device_properties(0).total_memory/2**30)"
execute_step(['-c',cuda_code])
print('Kernel Jupyter sin PyTorch:', 'torch' not in sys.modules)


RTX ENCONTRADA: NVIDIA GeForce RTX 4070 Ti SUPER, 16376 MiB, 591.86
CPU: AMD Ryzen 7 8700F 8-Core Processor             
Hilos lógicos: 16 RAM instalada GiB: 63.66
LLAMADA EXPLÍCITA: ['C:\\Python311\\python.exe', '-u', '-c', "import torch; print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda); assert torch.cuda.is_available(); gpu=torch.cuda.get_device_name(0); print('GPU CUDA:',gpu); assert '4070 Ti SUPER' in gpu; x=torch.ones((1,384),device='cuda'); print('Tensor real:',x.shape,x.device); print('VRAM GiB:',torch.cuda.get_device_properties(0).total_memory/2**30)"]


PyTorch: 2.5.1+cu121 CUDA: 12.1


GPU CUDA: NVIDIA GeForce RTX 4070 Ti SUPER


Tensor real: torch.Size([1, 384]) cuda:0


VRAM GiB: 15.99169921875


Proceso terminado; sus modelos y memoria quedaron liberados.


Kernel Jupyter sin PyTorch: True


## 2. DataFrames y etiquetas antes de entrenar
Se abre, muestra y libera un DataFrame por vez. Se conservan los IDs, URLs y `split_group` de procedencia. Los splits originales se agrupan por conversación y se quitan duplicados. Los títulos solo supervisan FAQ; los comentarios etiquetados por Gemini supervisan logro/sentimiento. No se inventan targets de logro/sentimiento para los títulos.


In [3]:
def show_dataframe(task):
    import pandas as pd
    pd.set_option('display.max_colwidth',None)
    df=pd.read_json(ROOT/'data'/f'{task}.jsonl',lines=True)
    cols=['id_mensaje','id_original','mensaje',task,'split','url']
    gemini=task+'_gemini'
    if gemini in df:cols.insert(4,gemini)
    print('ABRIR DF:',task,'filas/columnas:',df.shape)
    print('HEAD del DataFrame que entrena esta cabeza:')
    display(df[cols].head())
    print('CONTEO DE ETIQUETAS:');display(df[task].value_counts().sort_index().rename('filas').to_frame())
    print('SPLITS Y ETIQUETAS:');display(pd.crosstab(df['split'],df[task]))
    assert df.id_mensaje.is_unique
    assert df.groupby('split_group')['split'].nunique().max()==1,'Conversación repetida entre splits'
    assert set(df[task].unique()) <= ({0,.5,1} if task=='sentimiento' else {0,1})
    for value in sorted(df[task].unique()):
        print('EJEMPLOS target=',value)
        display(df.loc[df[task].eq(value),cols].head(3))
    if task=='sentimiento':
        print('ETIQUETAS ORIGINALES GEMINI:');display(df.sentimiento_gemini.value_counts().to_frame())
        print('EJEMPLOS MIXTOS, target 0.5:');display(df.loc[df.sentimiento_gemini.eq('mixto'),cols].head(3))
    del df;gc.collect()
    print('DataFrame liberado antes de abrir el siguiente.')


In [4]:
print('PASO 2 · es_faq')
show_dataframe('es_faq')

PASO 2 · es_faq


ABRIR DF: es_faq filas/columnas: (9782, 8)
HEAD del DataFrame que entrena esta cabeza:


,id_mensaje,id_original,mensaje,es_faq,split,url
0,1,es.stackoverflow:comment:1112204,Y probaste instalar los paquetes faltantes??,0,test,https://es.stackoverflow.com/posts/631747#comment1112204_631747
1,2,es.stackoverflow:comment:1112206,"Varias cosas.. Esto no es una red social. Arrobar a alguien que no participa en la publicación no sirve de nada porque no seran notificados. Despues, creo que tenes un error de concepto terrible, tu archivo se esta llamando a si mismo, con una llamada asincronica, que es basicamente que estas llamando a otro archivo? no tiene mucho sentido lo que estas planteando..",0,train,https://es.stackoverflow.com/posts/631739#comment1112206_631739
2,3,es.stackoverflow:comment:1112207,"Estás poniendo solo partes del código que no contienen errores. Tampoco pones el error real. Sin esa información, no podemos ayudarte.",0,train,https://es.stackoverflow.com/posts/631741#comment1112207_631741
3,4,es.stackoverflow:comment:1112208,"Si lo armas de una vez, tal vez llegue; inténtalo y también haz un var_dump(). No es pecado capital llamar al mismo archivo, se puede manejar sin problemas, aunque hasta el momento, tu código parece absurdo al hacer en el servidor algo que puede hacerse en el cliente, supongo -con optimismo- que librado el obstáculo hará algo más interesante.",0,train,https://es.stackoverflow.com/posts/631739#comment1112208_631739
4,5,es.stackoverflow:comment:1112209,Pero ¿por qué armas la sentencia en una cadena? si estas en t-sql mejor ejecuta directamente la sentencia parametrizando los valores. ¿El archivo binario esta en el servidor?.,0,train,https://es.stackoverflow.com/posts/631741#comment1112209_631741


CONTEO DE ETIQUETAS:


,filas
es_faq,
0,4782
1,5000


...

EJEMPLOS target= 0


,id_mensaje,id_original,mensaje,es_faq,split,url
0,1,es.stackoverflow:comment:1112204,Y probaste instalar los paquetes faltantes??,0,test,https://es.stackoverflow.com/posts/631747#comment1112204_631747
1,2,es.stackoverflow:comment:1112206,"Varias cosas.. Esto no es una red social. Arrobar a alguien que no participa en la publicación no sirve de nada porque no seran notificados. Despues, creo que tenes un error de concepto terrible, tu archivo se esta llamando a si mismo, con una llamada asincronica, que es basicamente que estas llamando a otro archivo? no tiene mucho sentido lo que estas planteando..",0,train,https://es.stackoverflow.com/posts/631739#comment1112206_631739
2,3,es.stackoverflow:comment:1112207,"Estás poniendo solo partes del código que no contienen errores. Tampoco pones el error real. Sin esa información, no podemos ayudarte.",0,train,https://es.stackoverflow.com/posts/631741#comment1112207_631741


EJEMPLOS target= 1


,id_mensaje,id_original,mensaje,es_faq,split,url
2340,2501,es.stackoverflow:question:625629,Mostrar +1 (1 visto) correctamente con localStorage al reproducir video,1,train,https://es.stackoverflow.com/questions/625629/mostrar-1-1-visto-correctamente-con-localstorage-al-reproducir-video
2341,2502,es.stackoverflow:question:625638,¿Como obtener los hijos inmediatos utilizando querySelectorAll?,1,train,https://es.stackoverflow.com/questions/625638/como-obtener-los-hijos-inmediatos-utilizando-queryselectorall
2342,2503,es.stackoverflow:question:625642,error en android studio en activity,1,test,https://es.stackoverflow.com/questions/625642/error-en-android-studio-en-activity


DataFrame liberado antes de abrir el siguiente.


In [5]:
print('PASO 2 · es_logro')
show_dataframe('es_logro')

PASO 2 · es_logro
ABRIR DF: es_logro filas/columnas: (4782, 12)
HEAD del DataFrame que entrena esta cabeza:


,id_mensaje,id_original,mensaje,es_logro,es_logro_gemini,split,url
0,1,es.stackoverflow:comment:1112204,Y probaste instalar los paquetes faltantes??,0,False,test,https://es.stackoverflow.com/posts/631747#comment1112204_631747
1,2,es.stackoverflow:comment:1112206,"Varias cosas.. Esto no es una red social. Arrobar a alguien que no participa en la publicación no sirve de nada porque no seran notificados. Despues, creo que tenes un error de concepto terrible, tu archivo se esta llamando a si mismo, con una llamada asincronica, que es basicamente que estas llamando a otro archivo? no tiene mucho sentido lo que estas planteando..",0,False,train,https://es.stackoverflow.com/posts/631739#comment1112206_631739
2,3,es.stackoverflow:comment:1112207,"Estás poniendo solo partes del código que no contienen errores. Tampoco pones el error real. Sin esa información, no podemos ayudarte.",0,False,train,https://es.stackoverflow.com/posts/631741#comment1112207_631741
3,4,es.stackoverflow:comment:1112208,"Si lo armas de una vez, tal vez llegue; inténtalo y también haz un var_dump(). No es pecado capital llamar al mismo archivo, se puede manejar sin problemas, aunque hasta el momento, tu código parece absurdo al hacer en el servidor algo que puede hacerse en el cliente, supongo -con optimismo- que librado el obstáculo hará algo más interesante.",0,False,train,https://es.stackoverflow.com/posts/631739#comment1112208_631739
4,5,es.stackoverflow:comment:1112209,Pero ¿por qué armas la sentencia en una cadena? si estas en t-sql mejor ejecuta directamente la sentencia parametrizando los valores. ¿El archivo binario esta en el servidor?.,0,False,train,https://es.stackoverflow.com/posts/631741#comment1112209_631741


CONTEO DE ETIQUETAS:


,filas
es_logro,
0,4536
1,246


SPLITS Y ETIQUETAS:


...

EJEMPLOS target= 0


,id_mensaje,id_original,mensaje,es_logro,es_logro_gemini,split,url
0,1,es.stackoverflow:comment:1112204,Y probaste instalar los paquetes faltantes??,0,False,test,https://es.stackoverflow.com/posts/631747#comment1112204_631747
1,2,es.stackoverflow:comment:1112206,"Varias cosas.. Esto no es una red social. Arrobar a alguien que no participa en la publicación no sirve de nada porque no seran notificados. Despues, creo que tenes un error de concepto terrible, tu archivo se esta llamando a si mismo, con una llamada asincronica, que es basicamente que estas llamando a otro archivo? no tiene mucho sentido lo que estas planteando..",0,False,train,https://es.stackoverflow.com/posts/631739#comment1112206_631739
2,3,es.stackoverflow:comment:1112207,"Estás poniendo solo partes del código que no contienen errores. Tampoco pones el error real. Sin esa información, no podemos ayudarte.",0,False,train,https://es.stackoverflow.com/posts/631741#comment1112207_631741


EJEMPLOS target= 1


,id_mensaje,id_original,mensaje,es_logro,es_logro_gemini,split,url
20,23,es.stackoverflow:comment:1112233,"Actualicé la respuesta, tal vez mejore.",1,True,train,https://es.stackoverflow.com/posts/631738#comment1112233_631738
27,31,es.stackoverflow:comment:1112250,"Disculpad que no haya respondido antes, al final opte por emplear un proyecto que sí tenía bien la importación de las librerías, de todos modos me suele dar bastantes quebraderos de cabeza la importación de esas librerías. Mil gracias",1,True,train,https://es.stackoverflow.com/posts/631731#comment1112250_631731
28,34,es.stackoverflow:comment:1112253,muchas gracias por el aporte. Da solución a mi problema.,1,True,train,https://es.stackoverflow.com/posts/631717#comment1112253_631717


DataFrame liberado antes de abrir el siguiente.


In [6]:
print('PASO 2 · sentimiento')
show_dataframe('sentimiento')

PASO 2 · sentimiento
ABRIR DF: sentimiento filas/columnas: (4781, 11)
HEAD del DataFrame que entrena esta cabeza:


,id_mensaje,id_original,mensaje,sentimiento,sentimiento_gemini,split,url
0,1,es.stackoverflow:comment:1112204,Y probaste instalar los paquetes faltantes??,0.5,neutro,test,https://es.stackoverflow.com/posts/631747#comment1112204_631747
1,2,es.stackoverflow:comment:1112206,"Varias cosas.. Esto no es una red social. Arrobar a alguien que no participa en la publicación no sirve de nada porque no seran notificados. Despues, creo que tenes un error de concepto terrible, tu archivo se esta llamando a si mismo, con una llamada asincronica, que es basicamente que estas llamando a otro archivo? no tiene mucho sentido lo que estas planteando..",0.0,negativo,train,https://es.stackoverflow.com/posts/631739#comment1112206_631739
2,3,es.stackoverflow:comment:1112207,"Estás poniendo solo partes del código que no contienen errores. Tampoco pones el error real. Sin esa información, no podemos ayudarte.",0.5,neutro,train,https://es.stackoverflow.com/posts/631741#comment1112207_631741
3,4,es.stackoverflow:comment:1112208,"Si lo armas de una vez, tal vez llegue; inténtalo y también haz un var_dump(). No es pecado capital llamar al mismo archivo, se puede manejar sin problemas, aunque hasta el momento, tu código parece absurdo al hacer en el servidor algo que puede hacerse en el cliente, supongo -con optimismo- que librado el obstáculo hará algo más interesante.",0.5,neutro,train,https://es.stackoverflow.com/posts/631739#comment1112208_631739
4,5,es.stackoverflow:comment:1112209,Pero ¿por qué armas la sentencia en una cadena? si estas en t-sql mejor ejecuta directamente la sentencia parametrizando los valores. ¿El archivo binario esta en el servidor?.,0.5,neutro,train,https://es.stackoverflow.com/posts/631741#comment1112209_631741


CONTEO DE ETIQUETAS:


,filas
sentimiento,
0.0,342
0.5,3915
1.0,524


SPLITS Y ETIQUETAS:


...

ETIQUETAS ORIGINALES GEMINI:


,count
sentimiento_gemini,
neutro,3904
positivo,524
negativo,342
mixto,11


EJEMPLOS MIXTOS, target 0.5:


,id_mensaje,id_original,mensaje,sentimiento,sentimiento_gemini,split,url
367,391,es.stackoverflow:comment:1112857,"lo demás esta bien explicado y atiende a los errores, pero me hubiera gustado algo mas completo que oriente hacia buenas practicas. XD",0.5,mixto,train,https://es.stackoverflow.com/posts/632161#comment1112857_632161
642,680,es.stackoverflow:comment:1113337,"@FranciscoIALover Gracias por la ayuda ! Quite el puerto :443, reedireccione al puerto 443 <VirtualHost *:80> ServerName lucasconde.ddns.net Redirect permanent / lucasconde.ddns.net </VirtualHost> y el SSL lo hice con Let's Encrypt que creo que es una authority (desde mi ignorancia) y sigo en la misma :/",0.5,mixto,train,https://es.stackoverflow.com/posts/632386#comment1113337_632386
864,914,es.stackoverflow:comment:1113720,"Listo, el copy del archivo ya funciona, tuve que cambiar into(""$buildDir/libs/config"") por into(""${buildDir}/libs/config""), pero el zip no se creo.",0.5,mixto,train,https://es.stackoverflow.com/posts/632619#comment1113720_632619


DataFrame liberado antes de abrir el siguiente.


In [7]:
print('PROCEDENCIA Y TRANSFORMACIONES:')
display(json.loads((ROOT/'data/provenance.json').read_text(encoding='utf-8')))
print('Sentimiento indeterminado: se conserva en sentimiento_revision.jsonl, fuera del entrenamiento.')
print('Un target diferente y un entrenamiento separado por cabeza.')


PROCEDENCIA Y TRANSFORMACIONES:


{'source': 'corpus_original.jsonl',
 'source_sha256': 'd39cde081d51d3d4f5be40af9f32b0c13629dc6c4a5b7aedde9c9f3a76ea5662',
 'faq': 'titulo Stack Exchange=1; comentario=0',
 'sentiment_mapping': {'negativo': 0,
  'neutro': 0.5,
  'positivo': 1,
  'mixto': 0.5},
 'no_quality_filter': True,
 'pending': 1,
 'rows': {'es_faq': 9782, 'es_logro': 4782, 'sentimiento': 4781}}

Sentimiento indeterminado: se conserva en sentimiento_revision.jsonl, fuera del entrenamiento.
Un target diferente y un entrenamiento separado por cabeza.


## 3. Encoder afinado, exportación y caché en SSD
El encoder v2 fue afinado con CUDA. Esta etapa usa sus pesos congelados. El pooling es media ponderada por `attention_mask`: produce 384 valores por mensaje, sin normalización adicional. En serving se procesan los **primeros 128 tokens**, batch 1, hasta 8,000 caracteres.

El encoder se exportó a FP32 (470.22 MB) y se cuantizó con `quantize_dynamic`, QInt8, `MatMul` y `Gather`, resultando en 118.05 MB. `Gather` permite reducir la tabla multilingüe. La representación final del mismo encoder usa pesos externos en SSD. INT8 aplica a pesos y operaciones seleccionadas: ReLU/Sigmoid y otras operaciones conservan valores flotantes.

La caché `.npy` se escribe de un mensaje por vez. `mmap` lee solamente los bloques usados por una cabeza. La caché no es necesaria para servir. Si cambia encoder, tokenizer, datos o política de tokens, hay que regenerarla antes de reentrenar.


In [8]:
print('REGISTRO REAL DEL AFINADO ANTERIOR DEL ENCODER:')
prior=ROOT/'artifacts_v2/mlp/training.json'
if prior.exists():
    history=json.loads(prior.read_text(encoding='utf-8'))
    display({k:v for k,v in history.items() if k in ['hardware','epochs','batch_size','history']})
    del history
show_source('cache_int8.py')
if RECREAR_CACHE or not (OUT/'embeddings_int8.npy').exists():execute_step([ROOT/'cache_int8.py'])
else:print('REUTILIZAR caché INT8 existente en SSD:',OUT/'embeddings_int8.npy')
print('Tamaño caché MB:',(OUT/'embeddings_int8.npy').stat().st_size/1e6)
cache_metadata=json.loads((OUT/'embeddings_index.json').read_text(encoding='utf-8'))
print('Filas:',len(cache_metadata['ids']),'dimensiones:',384,'precisión:',cache_metadata['precision'],'tokens:',cache_metadata['max_tokens'])
del cache_metadata;gc.collect()


REGISTRO REAL DEL AFINADO ANTERIOR DEL ENCODER:


{'hardware': {'gpu': 'NVIDIA GeForce RTX 4070 Ti SUPER',
  'vram_gib': 15.99169921875,
  'cpu_threads': 16,
  'torch_cpu_threads': 8,
  'ram_gib': 63.66278076171875,
  'torch': '2.5.1+cu121',
  'cuda': '12.1',
  'cpu': 'AMD Ryzen 7 8700F'},
 'epochs': 4,
 'batch_size': 128,
 'history': [{'epoch': 1,
   'train_loss': {'es_faq': 0.4526410420094767,
    'es_logro': 0.28630513769965016,
    'sentimiento': 0.697964366405241},
   'validation_loss': {'es_faq': 0.15408100181696366,
    'es_logro': 0.16010060533881187,
    'sentimiento': 0.694713905453682}},
  {'epoch': 2,
   'train_loss': {'es_faq': 0.09507259378029455,
    'es_logro': 0.15906138958469515,
    'sentimiento': 0.6919393693247149},
   'validation_loss': {'es_faq': 0.06791764290965334,
    'es_logro': 0.10900774784386158,
    'sentimiento': 0.6840747743844986}},
  {'epoch': 3,
   'train_loss': {'es_faq': 0.0428895216525322,
    'es_logro': 0.10250757057820598,
    'sentimiento': 0.6724429572782209},
   'validation_loss': {'es_faq'

IMPLEMENTACIÓN: cache_int8.py


"""Materializar embeddings INT8 de a uno en SSD; no importa torch ni pandas."""
import os,sys,json,time
from pathlib import Path
import numpy as np
ROOT=Path(__file__).resolve().parent
sys.path.insert(0,str(ROOT/'deps'))
from serve_mlp_int8 import session
OUT=ROOT/'artifacts_mlp_int8';OUT.mkdir(exist_ok=True)
source=ROOT/'data/es_faq.jsonl'
with source.open(encoding='utf-8') as stream:
    ids=[json.loads(line)['id_mensaje'] for line in stream]
lookup={v:i for i,v in enumerate(ids)}
from tokenizers import Tokenizer
directory=ROOT/'deploy_mlp_int8'
model_path=directory/'encoder.onnx'
tokenizer_path=directory/'tokenizer.json'
if not model_path.exists():
    model_path=ROOT/'artifacts_pca8/encoder/encoder_int8.onnx'
    tokenizer_path=ROOT/'artifacts_pca8/encoder/tokenizer.json'
print('ENCODER INT8 CPU -> SSD:',model_path,'sin Torch ni DataFrames',flush=True)
model=session(model_path)
inputs={v.name for v in model.get_inputs()}
tokenizer=Tokenizer.from_file(str(tokenizer_path));tokenizer.enable_truncation(max_length=128);tokenizer.no_padding()
cache=np.lib.format.open_memmap(OUT/'embeddings_int8.npy',mode='w+',dtype='float32',shape=(len(ids),384))
start=time.perf_counter()
with source.open(encoding='utf-8') as f:
    for count,line in enumerate(f,1):
        row=json.loads(line);tok=tokenizer.encode(row['mensaje'])
        feed={'input_ids':np.array([tok.ids],dtype=np.int64),'attention_mask':np.ones((1,len(tok.ids)),dtype=np.int64),'token_type_ids':np.array([tok.type_ids],dtype=np.int64)}
        cache[lookup[int(row['id_mensaje'])]]=model.run(None,{k:v for k,v in feed.items() if k in inputs})[0][0]
        if count%1000==0:print('Encoder INT8 CPU -> SSD:',count,'/',len(ids),flush=True)
cache.flush()
(OUT/'embeddings_index.json').write_text(json.dumps(dict(ids=ids,precision='int8',max_tokens=128,source=str(source),seconds=time.perf_counter()-start)),encoding='utf-8')
print('Embeddings INT8 guardados en SSD:',OUT/'embeddings_int8.npy',flush=True)

LLAMADA EXPLÍCITA: ['C:\\Python311\\python.exe', '-u', 'C:\\Users\\saul_\\Documents\\Codex\\2026-10-04\\github_delivery\\data_science\\2026-10-04\\cache_int8.py']


...

Encoder INT8 CPU -> SSD: 9000 / 9782


Embeddings INT8 guardados en SSD: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\embeddings_int8.npy


Proceso terminado; sus modelos y memoria quedaron liberados.


Tamaño caché MB: 15.02528
Filas: 9782 dimensiones: 384 precisión: int8 tokens: 128


0

## 4. Definir y entrenar tres cabezas en RTX, por separado
Arquitecturas: pequeña `[384,16,4,1]`, profunda `[384,64,32,16,4,1]`. Cada cabeza tiene su propio optimizador, target y checkpoint. ReLU tras cada capa oculta. Durante el entrenamiento se usa BCEWithLogitsLoss para estabilidad; la sigmoide se aplica en evaluación y queda en ONNX.

AdamW: lr=0.001, weight_decay=0.001; batches=128; seed=42; hasta 100 épocas; parada tras 15 sin mejorar BCE de validation. Solo train ajusta pesos. Cada proceso muestra `features cuda:0`, `target cuda:0` y `head cuda:0`, guarda el mejor checkpoint y exporta su cabeza FP32 y después INT8, y termina antes de la próxima.

Con REENTRENAR=False se muestran los registros **de entrenamientos realmente ejecutados**. Con True se repiten las seis ejecuciones de forma secuencial. El proceso PyTorch puede superar 1 GB en la PC local; no se carga en el runtime OCI.


In [9]:
show_source('train_mlp_head.py')

IMPLEMENTACIÓN: train_mlp_head.py


"""Entrenar y exportar UNA cabeza MLP por proceso. Embeddings INT8 en SSD, entrenamiento CUDA."""
import os,sys,json,time,gc,hashlib
from pathlib import Path
ROOT=Path(__file__).resolve().parent
sys.path.insert(0,str(ROOT/'deps'))
OUT=ROOT/'artifacts_mlp_int8'
TASKS=('es_faq','es_logro','sentimiento')
DIMS={'small':[384,16,4,1],'deep':[384,64,32,16,4,1]}

def train(task,architecture):
    import numpy as np,pandas as pd,torch
    from sklearn.metrics import f1_score,mean_absolute_error,confusion_matrix
    import onnx,onnxruntime as ort
    from onnxruntime.quantization import quantize_dynamic,QuantType
    assert torch.cuda.is_available(),'CUDA requerida'
    gpu=torch.cuda.get_device_name(0);assert '4070 Ti SUPER' in gpu,gpu
    print('[1] RTX DETECTADA:',gpu,flush=True)
    print('PyTorch',torch.__version__,'CUDA',torch.version.cuda,'VRAM GiB',torch.cuda.get_device_properties(0).total_memory/2**30,flush=True)
    torch.set_num_threads(8);torch.manual_seed(42);torch.cuda.manual_seed_all(42)
    df=pd.read_json(ROOT/'data'/f'{task}.jsonl',lines=True)
    print('[2] DATAFRAME',task,'shape=',df.shape,flush=True)
    label=task+'_gemini'
    columns=['id_mensaje','mensaje',task,'split']+([label] if label in df else [])
    print(df[columns].head().to_string(index=False),flush=True)
    print('Etiquetas y soporte:',df[task].value_counts().sort_index().to_dict(),flush=True)
    for value in sorted(df[task].unique()):print('Ejemplo target',value,':',df[df[task].eq(value)].iloc[0].mensaje,flush=True)
    ids=json.loads((OUT/'embeddings_index.json').read_text(encoding='utf-8'))['ids'];lookup={v:i for i,v in enumerate(ids)}
    indices=np.array([lookup[int(i)] for i in df.id_mensaje])
    cache=np.load(OUT/'embeddings_int8.npy',mmap_mode='r')
    y=df[task].to_numpy(dtype=np.float32)
    rows={split:np.flatnonzero(df.split.eq(split).to_numpy()) for split in ['train','validation','test']}
    print('[3] SSD mmap',cache.shape,'train/validation/test',{k:len(v) for k,v in rows.items()},flush=True)
    layers=[]
    for i,(a,b) in enumerate(zip(DIMS[architecture],DIMS[architecture][1:])):
        layers.append(torch.nn.Linear(a,b))
        if i<len(DIMS[architecture])-2:layers.append(torch.nn.ReLU())
    model=torch.nn.Sequential(*layers).cuda()
    optimizer=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.001)
    criterion=torch.nn.BCEWithLogitsLoss()
    out=OUT/architecture/task;out.mkdir(parents=True,exist_ok=True)
    print('[4] MODELO:',model,flush=True)
    print('Parámetros:',sum(p.numel() for p in model.parameters()),'device:',next(model.parameters()).device,flush=True)
    best=float('inf');stale=0;history=[];rng=np.random.default_rng(42);start=time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    @torch.inference_mode()
    def probabilities(which):
        model.eval();values=[]
        for offset in range(0,len(which),256):
            ix=which[offset:offset+256]
            x=torch.tensor(np.asarray(cache[indices[ix]]),device='cuda')
            values.append(model(x).sigmoid().flatten().cpu().numpy())
        return np.concatenate(values)
    for epoch in range(100):
        model.train();losses=[]
        shuffled=rng.permutation(rows['train'])
        for offset in range(0,len(shuffled),128):
            ix=shuffled[offset:offset+128]
            x=torch.tensor(np.asarray(cache[indices[ix]]),device='cuda')
            target=torch.tensor(y[ix,None],device='cuda')
            if epoch==0 and offset==0:
                print('CHECK CUDA: features',x.device,x.shape,'target',target.device,'head',next(model.parameters()).device,flush=True)
            optimizer.zero_grad(set_to_none=True)
            loss=criterion(model(x),target);loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            optimizer.step();losses.append(float(loss))
        pv=probabilities(rows['validation'])
        tv=y[rows['validation']]
        vl=float(np.mean(-(tv*np.log(np.clip(pv,1e-7,1-1e-7))+(1-tv)*np.lo

In [10]:
def one_head(architecture,task):
    path=OUT/architecture/task/'report.json'
    print('CABEZAS SECUENCIALES ·',architecture,task)
    if REENTRENAR or not path.exists():execute_step([ROOT/'train_mlp_head.py',task,architecture])
    else:
        print('Cabeza ya entrenada; registro guardado:',path)
        logfile=OUT/'logs'/f'{architecture}_{task}.log'
        if logfile.exists():
            text=logfile.read_text(encoding='utf-8')
            if '[9] FIN.' in text:print(text)
            else:print('El log conserva el intento interrumpido. El reintento completó; abajo se muestra su report.json y toda su historia.')
    report=json.loads(path.read_text(encoding='utf-8'))
    print('RTX realmente usada:',report['gpu'],'CHECK CUDA:',report['cuda_checks'])
    print('Dimensiones:',report['dims'],'parámetros:',report['parameters'],'VRAM pico MB:',report['peak_training_vram_mb'])
    print('Tiempo segundos:',report['train_seconds'],'épocas:',report['epochs'])
    print('TODAS LAS ÉPOCAS:');display(report['history'])
    print('VALIDATION:');display(report['validation'])
    print('TEST INT8:');display(report['test'])
    print('Error CUDA / ONNX FP32:',report['fp32_parity_error'])
    print('Cortes elegidos SOLO en validation:',report['threshold'])
    print('ONNX INT8 en SSD:',path.parent/'head_int8.onnx','bytes:',(path.parent/'head_int8.onnx').stat().st_size)
    del report;gc.collect()
    print('FIN DE CABEZA. No hay modelo de entrenamiento vivo en el kernel.')


In [11]:
one_head('small','es_faq')

CABEZAS SECUENCIALES · small es_faq
Cabeza ya entrenada; registro guardado: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\small\es_faq\report.json
[1] RTX DETECTADA: NVIDIA GeForce RTX 4070 Ti SUPER
PyTorch 2.5.1+cu121 CUDA 12.1 VRAM GiB 15.99169921875
[2] DATAFRAME es_faq shape= (9782, 8)
 id_mensaje                                                                                                                                                                                                                                                                                                                                                                         mensaje  es_faq split
          1                                                                                                                                                                                                                                                         

[{'epoch': 1,
  'train_loss': 0.2817457402425428,
  'validation_loss': 0.09243489801883698},
 {'epoch': 2,
  'train_loss': 0.030794715659031944,
  'validation_loss': 0.05285729840397835},
 {'epoch': 3,
  'train_loss': 0.015087608893912646,
  'validation_loss': 0.05366174876689911},
 {'epoch': 4,
  'train_loss': 0.01307845048441161,
  'validation_loss': 0.05428389832377434},
 {'epoch': 5,
  'train_loss': 0.01244953325799396,
  'validation_loss': 0.0554245226085186},
 {'epoch': 6,
  'train_loss': 0.012271513721169603,
  'validation_loss': 0.056187473237514496},
 {'epoch': 7,
  'train_loss': 0.012067593793730222,
  'validation_loss': 0.05600620433688164},
 {'epoch': 8,
  'train_loss': 0.012352053154890815,
  'validation_loss': 0.05581890046596527},
 {'epoch': 9,
  'train_loss': 0.011871717958485769,
  'validation_loss': 0.0551835335791111},
 {'epoch': 10,
  'train_loss': 0.011743418748430427,
  'validation_loss': 0.054966993629932404},
 {'epoch': 11,
  'train_loss': 0.01165101790027092,
 

VALIDATION:


{'f1': 0.9862433862433863,
 'f1_macro': 0.9867177522349937,
 'support': 980,
 'positives': 468}

TEST INT8:


{'f1': 0.9796334012219959,
 'f1_macro': 0.9776807782809009,
 'support': 903,
 'positives': 485}

Error CUDA / ONNX FP32: 1.1920928955078125e-07
Cortes elegidos SOLO en validation: 0.11
ONNX INT8 en SSD: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\small\es_faq\head_int8.onnx bytes: 11193
FIN DE CABEZA. No hay modelo de entrenamiento vivo en el kernel.


In [12]:
one_head('small','es_logro')

CABEZAS SECUENCIALES · small es_logro
Cabeza ya entrenada; registro guardado: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\small\es_logro\report.json
[1] RTX DETECTADA: NVIDIA GeForce RTX 4070 Ti SUPER
PyTorch 2.5.1+cu121 CUDA 12.1 VRAM GiB 15.99169921875
[2] DATAFRAME es_logro shape= (4782, 12)
 id_mensaje                                                                                                                                                                                                                                                                                                                                                                         mensaje  es_logro split  es_logro_gemini
          1                                                                                                                                                                                                                               

[{'epoch': 1,
  'train_loss': 0.2281831696389183,
  'validation_loss': 0.127455472946167},
 {'epoch': 2,
  'train_loss': 0.08756490087797565,
  'validation_loss': 0.10529224574565887},
 {'epoch': 3,
  'train_loss': 0.06714973971247673,
  'validation_loss': 0.10289250314235687},
 {'epoch': 4,
  'train_loss': 0.062484187344389576,
  'validation_loss': 0.10373452305793762},
 {'epoch': 5,
  'train_loss': 0.0478359911470644,
  'validation_loss': 0.10639874637126923},
 {'epoch': 6,
  'train_loss': 0.04198723193258047,
  'validation_loss': 0.11387210339307785},
 {'epoch': 7,
  'train_loss': 0.039000042744221225,
  'validation_loss': 0.11973606795072556},
 {'epoch': 8,
  'train_loss': 0.03711915014672183,
  'validation_loss': 0.12388233840465546},
 {'epoch': 9,
  'train_loss': 0.035929789045645345,
  'validation_loss': 0.126581609249115},
 {'epoch': 10,
  'train_loss': 0.038110042802028114,
  'validation_loss': 0.12859240174293518},
 {'epoch': 11,
  'train_loss': 0.035061847538717335,
  'valid

VALIDATION:


{'f1': 0.5957446808510638,
 'f1_macro': 0.7881486966179576,
 'support': 512,
 'positives': 22}

TEST INT8:


{'f1': 0.4444444444444444,
 'f1_macro': 0.7097222222222221,
 'support': 418,
 'positives': 18}

Error CUDA / ONNX FP32: 2.9802322387695312e-08
Cortes elegidos SOLO en validation: 0.3
ONNX INT8 en SSD: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\small\es_logro\head_int8.onnx bytes: 11193
FIN DE CABEZA. No hay modelo de entrenamiento vivo en el kernel.


In [13]:
one_head('small','sentimiento')

CABEZAS SECUENCIALES · small sentimiento
Cabeza ya entrenada; registro guardado: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\small\sentimiento\report.json
El log conserva el intento interrumpido. El reintento completó; abajo se muestra su report.json y toda su historia.
RTX realmente usada: NVIDIA GeForce RTX 4070 Ti SUPER CHECK CUDA: True
Dimensiones: [384, 16, 4, 1] parámetros: 6233 VRAM pico MB: 18.628096
Tiempo segundos: 2.445186900004046 épocas: 52
TODAS LAS ÉPOCAS:


[{'epoch': 1,
  'train_loss': 0.7157783565982696,
  'validation_loss': 0.6978591680526733},
 {'epoch': 2,
  'train_loss': 0.6905707563123396,
  'validation_loss': 0.6791905760765076},
 {'epoch': 3,
  'train_loss': 0.6685751772695973,
  'validation_loss': 0.6627777814865112},
 {'epoch': 4,
  'train_loss': 0.6558449191431845,
  'validation_loss': 0.6589468121528625},
 {'epoch': 5,
  'train_loss': 0.6521173292590726,
  'validation_loss': 0.6571425199508667},
 {'epoch': 6,
  'train_loss': 0.6477403160064451,
  'validation_loss': 0.6561852693557739},
 {'epoch': 7,
  'train_loss': 0.6474188104752572,
  'validation_loss': 0.6557239294052124},
 {'epoch': 8,
  'train_loss': 0.6482248440865548,
  'validation_loss': 0.6551494598388672},
 {'epoch': 9,
  'train_loss': 0.6477007404450448,
  'validation_loss': 0.6546967029571533},
 {'epoch': 10,
  'train_loss': 0.6447391990692385,
  'validation_loss': 0.6537009477615356},
 {'epoch': 11,
  'train_loss': 0.6466567227917333,
  'validation_loss': 0.65359

VALIDATION:


{'f1_macro': 0.667444586159438,
 'mae': 0.08575010299682617,
 'confusion_matrix': [[16, 22, 0], [37, 373, 9], [2, 14, 39]],
 'support': 512}

TEST INT8:


{'f1_macro': 0.6769588083166886,
 'mae': 0.08472999185323715,
 'confusion_matrix': [[15, 8, 0], [31, 305, 10], [2, 15, 32]],
 'support': 418}

Error CUDA / ONNX FP32: 1.1920928955078125e-07
Cortes elegidos SOLO en validation: [0.42499999999999993, 0.7500000000000002]
ONNX INT8 en SSD: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\small\sentimiento\head_int8.onnx bytes: 11193


FIN DE CABEZA. No hay modelo de entrenamiento vivo en el kernel.

In [14]:
one_head('deep','es_faq')

CABEZAS SECUENCIALES · deep es_faq
Cabeza ya entrenada; registro guardado: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\deep\es_faq\report.json
[1] RTX DETECTADA: NVIDIA GeForce RTX 4070 Ti SUPER
PyTorch 2.5.1+cu121 CUDA 12.1 VRAM GiB 15.99169921875
[2] DATAFRAME es_faq shape= (9782, 8)
 id_mensaje                                                                                                                                                                                                                                                                                                                                                                         mensaje  es_faq split
          1                                                                                                                                                                                                                                                           

[{'epoch': 1,
  'train_loss': 0.34666226527863936,
  'validation_loss': 0.07697533071041107},
 {'epoch': 2,
  'train_loss': 0.016775008315415754,
  'validation_loss': 0.06235611066222191},
 {'epoch': 3,
  'train_loss': 0.013922140028538001,
  'validation_loss': 0.06041303649544716},
 {'epoch': 4,
  'train_loss': 0.013385241443756968,
  'validation_loss': 0.05801961198449135},
 {'epoch': 5,
  'train_loss': 0.01237329741519305,
  'validation_loss': 0.0552702471613884},
 {'epoch': 6,
  'train_loss': 0.012309365342521379,
  'validation_loss': 0.05407183989882469},
 {'epoch': 7,
  'train_loss': 0.012156877616014812,
  'validation_loss': 0.05144534632563591},
 {'epoch': 8,
  'train_loss': 0.011985362970247684,
  'validation_loss': 0.058300215750932693},
 {'epoch': 9,
  'train_loss': 0.012358116146530054,
  'validation_loss': 0.0515754297375679},
 {'epoch': 10,
  'train_loss': 0.011623083729289411,
  'validation_loss': 0.049297232180833817},
 {'epoch': 11,
  'train_loss': 0.011584274971204239

VALIDATION:


{'f1': 0.9873150105708245,
 'f1_macro': 0.9877403455220987,
 'support': 980,
 'positives': 468}

TEST INT8:


{'f1': 0.9776422764227642,
 'f1_macro': 0.9754391430775622,
 'support': 903,
 'positives': 485}

Error CUDA / ONNX FP32: 5.960464477539063e-08
Cortes elegidos SOLO en validation: 0.060000000000000005
ONNX INT8 en SSD: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\deep\es_faq\head_int8.onnx bytes: 36350
FIN DE CABEZA. No hay modelo de entrenamiento vivo en el kernel.


In [15]:
one_head('deep','es_logro')

CABEZAS SECUENCIALES · deep es_logro
Cabeza ya entrenada; registro guardado: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\deep\es_logro\report.json
[1] RTX DETECTADA: NVIDIA GeForce RTX 4070 Ti SUPER
PyTorch 2.5.1+cu121 CUDA 12.1 VRAM GiB 15.99169921875
[2] DATAFRAME es_logro shape= (4782, 12)
 id_mensaje                                                                                                                                                                                                                                                                                                                                                                         mensaje  es_logro split  es_logro_gemini
          1                                                                                                                                                                                                                                 

[{'epoch': 1,
  'train_loss': 0.4478697042551733,
  'validation_loss': 0.11580726504325867},
 {'epoch': 2,
  'train_loss': 0.059303022139976104,
  'validation_loss': 0.11494192481040955},
 {'epoch': 3,
  'train_loss': 0.04865871465975238,
  'validation_loss': 0.1207156702876091},
 {'epoch': 4,
  'train_loss': 0.05308765969088962,
  'validation_loss': 0.1224365159869194},
 {'epoch': 5,
  'train_loss': 0.03928744300238548,
  'validation_loss': 0.12306445837020874},
 {'epoch': 6,
  'train_loss': 0.03723631585918127,
  'validation_loss': 0.13593360781669617},
 {'epoch': 7,
  'train_loss': 0.037048256775784875,
  'validation_loss': 0.13574977219104767},
 {'epoch': 8,
  'train_loss': 0.036739854814274414,
  'validation_loss': 0.13419915735721588},
 {'epoch': 9,
  'train_loss': 0.03687481593196055,
  'validation_loss': 0.1354585587978363},
 {'epoch': 10,
  'train_loss': 0.03866264545508931,
  'validation_loss': 0.13327066600322723},
 {'epoch': 11,
  'train_loss': 0.03728715344397877,
  'valid

VALIDATION:


{'f1': 0.625, 'f1_macro': 0.8032786885245902, 'support': 512, 'positives': 22}

TEST INT8:


{'f1': 0.43243243243243246,
 'f1_macro': 0.7030747894327369,
 'support': 418,
 'positives': 18}

Error CUDA / ONNX FP32: 5.960464477539063e-08
Cortes elegidos SOLO en validation: 0.41000000000000003
ONNX INT8 en SSD: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\deep\es_logro\head_int8.onnx bytes: 36350
FIN DE CABEZA. No hay modelo de entrenamiento vivo en el kernel.


In [16]:
one_head('deep','sentimiento')

CABEZAS SECUENCIALES · deep sentimiento
Cabeza ya entrenada; registro guardado: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\deep\sentimiento\report.json
[1] RTX DETECTADA: NVIDIA GeForce RTX 4070 Ti SUPER
PyTorch 2.5.1+cu121 CUDA 12.1 VRAM GiB 15.99169921875
[2] DATAFRAME sentimiento shape= (4781, 11)
 id_mensaje                                                                                                                                                                                                                                                                                                                                                                         mensaje  sentimiento split sentimiento_gemini
          1                                                                                                                                                                                                                   

[{'epoch': 1,
  'train_loss': 0.6813910757341692,
  'validation_loss': 0.6804324984550476},
 {'epoch': 2,
  'train_loss': 0.6694916063739408,
  'validation_loss': 0.6638191938400269},
 {'epoch': 3,
  'train_loss': 0.6517330119686742,
  'validation_loss': 0.6557291746139526},
 {'epoch': 4,
  'train_loss': 0.6469386969843218,
  'validation_loss': 0.653784990310669},
 {'epoch': 5,
  'train_loss': 0.6452970312487695,
  'validation_loss': 0.6527921557426453},
 {'epoch': 6,
  'train_loss': 0.6414559272027784,
  'validation_loss': 0.6520400047302246},
 {'epoch': 7,
  'train_loss': 0.6412762249669721,
  'validation_loss': 0.6513764262199402},
 {'epoch': 8,
  'train_loss': 0.6415370971925797,
  'validation_loss': 0.6514064073562622},
 {'epoch': 9,
  'train_loss': 0.6415618946475368,
  'validation_loss': 0.6511666774749756},
 {'epoch': 10,
  'train_loss': 0.6382414141008931,
  'validation_loss': 0.6535614728927612},
 {'epoch': 11,
  'train_loss': 0.6417658540510363,
  'validation_loss': 0.652531

VALIDATION:


{'f1_macro': 0.643530261396463,
 'mae': 0.09218735247850418,
 'confusion_matrix': [[11, 27, 0], [29, 374, 16], [2, 10, 43]],
 'support': 512}

TEST INT8:


{'f1_macro': 0.6683159545601995,
 'mae': 0.0913723036646843,
 'confusion_matrix': [[12, 11, 0], [20, 310, 16], [2, 14, 33]],
 'support': 418}

Error CUDA / ONNX FP32: 5.960464477539063e-08
Cortes elegidos SOLO en validation: [0.3999999999999999, 0.6500000000000001]
ONNX INT8 en SSD: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\artifacts_mlp_int8\deep\sentimiento\head_int8.onnx bytes: 36350


FIN DE CABEZA. No hay modelo de entrenamiento vivo en el kernel.


## 5. Comparar y documentar la decisión
Se usa el F1 macro medio de las tres tareas en validation. La regla acepta la profunda si pierde como máximo 0.005 frente a la pequeña, conforme a la preferencia por la profunda cuando la diferencia es pequeña y el costo mínimo. Test informa el resultado y no elige arquitectura.

La profunda no demostró superioridad en esta prueba: ambas están próximas y la pequeña también es válida. El costo extra de estas tres cabezas es aproximadamente 75 KB. La RAM del encoder/runtime es mucho más relevante.

Para FAQ y logro se informa F1 positivo; sentimiento se agrupa en negativo/central/positivo y se informa macro F1 y MAE. Neutro/mixto comparten la clase central. Los cortes de predicción se ajustan solo en validation; los targets 0/0.5/1 no se alteran.


In [17]:
execute_step([ROOT/'run_sequential_mlp.py'])
summary=json.loads((OUT/'comparison.json').read_text(encoding='utf-8'))
import pandas as pd
rows=[]
for architecture,tasks in summary['architectures'].items():
    for task,r in tasks.items():
        rows.append(dict(arquitectura=architecture,tarea=task,dimensiones=str(r['dims']),parametros=r['parameters'],
                         validation_macro_f1=r['validation']['f1_macro'],test_f1=r['test'].get('f1',r['test']['f1_macro']),
                         segundos=r['train_seconds'],onnx_int8_bytes=(OUT/architecture/task/'head_int8.onnx').stat().st_size))
display(pd.DataFrame(rows))
print('F1 macro promedio validation:',summary['validation_macro_f1'])
print('SELECCIÓN:',summary['selected'],summary['selection_rule'])
del rows,summary;gc.collect()


LLAMADA EXPLÍCITA: ['C:\\Python311\\python.exe', '-u', 'C:\\Users\\saul_\\Documents\\Codex\\2026-10-04\\github_delivery\\data_science\\2026-10-04\\run_sequential_mlp.py']


Reutilizar cabeza ya guardada: small es_faq


Reutilizar cabeza ya guardada: small es_logro


Reutilizar cabeza ya guardada: small sentimiento


Reutilizar cabeza ya guardada: deep es_faq


...

SELECCION deep {'small': 0.8141036783374632, 'deep': 0.8115164318143839}


Proceso terminado; sus modelos y memoria quedaron liberados.


,arquitectura,tarea,dimensiones,parametros,validation_macro_f1,test_f1,segundos,onnx_int8_bytes
0,small,es_faq,"[384, 16, 4, 1]",6233,0.986718,0.979633,1.758701,11193
1,small,es_logro,"[384, 16, 4, 1]",6233,0.788149,0.444444,1.063559,11193
2,small,sentimiento,"[384, 16, 4, 1]",6233,0.667445,0.676959,2.445187,11193
3,deep,es_faq,"[384, 64, 32, 16, 4, 1]",27321,0.987740,0.977642,3.075123,36350
4,deep,es_logro,"[384, 64, 32, 16, 4, 1]",27321,0.803279,0.432432,1.207525,36350
5,deep,sentimiento,"[384, 64, 32, 16, 4, 1]",27321,0.643530,0.668316,1.641444,36350


F1 macro promedio validation: {'small': 0.8141036783374632, 'deep': 0.8115164318143839}
SELECCIÓN: deep F1 macro medio de validation. Deep si no pierde más de 0.005 frente a small; los pesos siguen siendo pequeños. Test no selecciona.


0

## 6. Guardar paquete ONNX portable y comprobar operadores
Un encoder compartido, tokenizer y tres archivos de cabezas. Los pesos externos ONNX se dividen en archivos de menos de 100 MB para GitHub; deben permanecer junto al grafo encoder.onnx. No se duplican los pesos por cabeza. El código conserva ReLU intermedias y Sigmoid final y comprueba presencia de operaciones enteras.


In [18]:
show_source('package_mlp.py')
if REENTRENAR or not (DEPLOY/'config.json').exists():execute_step([ROOT/'package_mlp.py'])
else:print('Paquete final ya escrito en SSD:',DEPLOY)
config=json.loads((DEPLOY/'config.json').read_text(encoding='utf-8'))
print('Arquitectura final:',config['dims'],'precisión:',config['precision'])
display(pd.DataFrame([dict(archivo=k,bytes=v['bytes']) for k,v in config['files'].items()]))
print('Total MB en SSD:',sum(v['bytes'] for v in config['files'].values())/1e6)
del config;gc.collect()


IMPLEMENTACIÓN: package_mlp.py


"""Paquete portable: un encoder compartido y tres cabezas, pesos ONNX en SSD."""
import json,shutil,hashlib
from pathlib import Path
import onnx
from onnx.external_data_helper import set_external_data
ROOT=Path(__file__).resolve().parent
OUT=ROOT/'artifacts_mlp_int8'
DEPLOY=ROOT/'deploy_mlp_int8'

def main():
    summary=json.loads((OUT/'comparison.json').read_text(encoding='utf-8'))
    arch=summary['selected'];reports=summary['architectures'][arch]
    DEPLOY.mkdir(exist_ok=True);(DEPLOY/'heads').mkdir(exist_ok=True)
    # GitHub admite cada archivo de pesos: ningún tensor individual llega a 100 MB.
    source=ROOT/'artifacts_pca8/encoder/encoder_int8.onnx'
    if not source.exists():source=DEPLOY/'encoder.onnx'
    print('UN encoder fuente:',source,flush=True)
    model=onnx.load(str(source));limit=100_000_000;index=0;offset=0;stream=None
    try:
        for tensor in model.graph.initializer:
            if not tensor.HasField('raw_data'):continue
            data=tensor.raw_data
            assert len(data)<=limit,(tensor.name,len(data))
            if stream is None or offset+len(data)>limit:
                if stream:stream.close()
                name=f'weights-{index:03d}.bin';index+=1;offset=0
                stream=(DEPLOY/name).open('wb')
            stream.write(data)
            set_external_data(tensor,location=name,offset=offset,length=len(data))
            tensor.ClearField('raw_data');offset+=len(data)
    finally:
        if stream:stream.close()
    (DEPLOY/'encoder.onnx').write_bytes(model.SerializeToString())
    del model
    onnx.checker.check_model(str(DEPLOY/'encoder.onnx'))
    tokenizer_source=ROOT/'artifacts_pca8/encoder/tokenizer.json'
    if tokenizer_source.exists():shutil.copy2(tokenizer_source,DEPLOY/'tokenizer.json')
    else:assert (DEPLOY/'tokenizer.json').exists()
    for task in reports:
        shutil.copy2(OUT/arch/task/'head_int8.onnx',DEPLOY/'heads'/f'{task}.onnx')
        head=onnx.load(str(DEPLOY/'heads'/f'{task}.onnx'))
        ops=[n.op_type for n in head.graph.node]
        assert 'Sigmoid' in ops and ops.count('Relu')==len(reports[task]['dims'])-2
        assert 'MatMulInteger' in ops,ops
        print(task,'dimensiones:',reports[task]['dims'],'operadores:',sorted(set(ops)),flush=True)
    files={str(p.relative_to(DEPLOY)).replace('\\','/'):{'bytes':p.stat().st_size,'sha256':hashlib.file_digest(p.open('rb'),'sha256').hexdigest()} for p in DEPLOY.rglob('*') if p.is_file() and p.name!='config.json'}
    config=dict(architecture=arch,dims=reports['es_faq']['dims'],tasks=list(reports),thresholds={t:r['threshold'] for t,r in reports.items()},
        encoder='MiniLM multilingual L12 v2, afinado v2 en RTX, congelado en esta comparación',
        source_revision='e8f8c211226b894fcb81acc59f3b34ba3efd5f42',precision='MatMul/Gather encoder INT8; MatMul cabezas INT8; activaciones ReLU/Sigmoid flotantes',
        max_tokens=128,max_chars=8000,batch=1,workers=1,process_budget_bytes=1_000_000_000,application_target_bytes=700_000_000,
        sentiment_mapping={'negativo':0,'neutro':.5,'mixto':.5,'positivo':1},files=files)
    (DEPLOY/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
    print('PAQUETE EN SSD:',DEPLOY,'MB:',sum(v['bytes'] for v in files.values())/1e6,flush=True)
    for name,value in files.items():print(name,value['bytes'],'bytes',flush=True)

if __name__=='__main__':main()

Paquete final ya escrito en SSD: C:\Users\saul_\Documents\Codex\2026-10-04\github_delivery\data_science\2026-10-04\deploy_mlp_int8
Arquitectura final: [384, 64, 32, 16, 4, 1] precisión: MatMul/Gather encoder INT8; MatMul cabezas INT8; activaciones ReLU/Sigmoid flotantes


,archivo,bytes
0,encoder.onnx,377779
1,tokenizer.json,17082833
2,weights-000.bin,99993216
3,weights-001.bin,17694720
4,heads/es_faq.onnx,36350
5,heads/es_logro.onnx,36350
6,heads/sentimiento.onnx,36350


Total MB en SSD: 135.257598


0

## 7. F1 del paquete real, batch 1 y límites del contrato
Se evalúa el encoder INT8 portable y cada cabeza INT8 de una en una con todos los ejemplos test. Se compara el embedding contra la caché. La cuantización dinámica puede depender del batch: se reajustan los cortes usando exclusivamente validation con batch 1 y se miden después los ejemplos test con el mismo batch de producción.

El score de sentimiento sigue siendo un índice continuo entre 0 y 1. Sus cortes solo permiten calcular F1 de tres clases; no cambian el JSON de salida ni garantizan calibración de probabilidades. La cabeza logro tiene pocos positivos: se muestran soporte y matriz de confusión para interpretar F1.


In [19]:
show_source('evaluate_mlp_package.py')
if REVALIDAR or REENTRENAR or not (OUT/'package_evaluation.json').exists():execute_step([ROOT/'evaluate_mlp_package.py'])
evaluation=json.loads((OUT/'package_evaluation.json').read_text(encoding='utf-8'))
print('Máximo error embedding caché / paquete:',evaluation['max_embedding_error_vs_cache'])
print('Alcance:',evaluation['scope'])
for task,record in evaluation['tasks'].items():
    print('TAREA:',task,'cortes validation batch 1:',record['threshold'])
    print('VALIDATION:');display(record['validation_batch1'])
    print('TEST:');display(record['test_batch1'])
    print('Truncados test:',record['truncated_test'],'rechazados test:',record['rejected_test'])
del evaluation;gc.collect()


IMPLEMENTACIÓN: evaluate_mlp_package.py


"""Validar paquete portable extremo a extremo, batch 1, una cabeza por vez."""
import sys,json,gc,time
from pathlib import Path
import numpy as np
from serve_mlp_int8 import Predictor,session,TASKS
ROOT=Path(__file__).resolve().parent
OUT=ROOT/'artifacts_mlp_int8'

def f1s(truth,pred,nclasses):
    matrix=np.zeros((nclasses,nclasses),dtype=int)
    for a,b in zip(truth,pred):matrix[int(a),int(b)]+=1
    values=[]
    for c in range(nclasses):
        tp=int(matrix[c,c]);den=int(matrix[c,:].sum()+matrix[:,c].sum())
        values.append(2*tp/den if den else 0.)
    return matrix,values

def categories(x,cuts):return np.where(x<cuts[0],0,np.where(x>cuts[1],2,1))

def metrics(task,y,p,cut):
    if task=='sentimiento':
        matrix,values=f1s(categories(y,(.25,.75)),categories(p,cut),3)
        return dict(f1_macro=float(np.mean(values)),mae=float(np.mean(np.abs(y-p))),f1_per_class=values,confusion_matrix=matrix.tolist(),support=len(y))
    matrix,values=f1s(y,p>=cut,2)
    return dict(f1=values[1],f1_macro=float(np.mean(values)),f1_per_class=values,confusion_matrix=matrix.tolist(),support=len(y),positives=int(y.sum()))

def main():
    predictor=Predictor();cache=np.load(OUT/'embeddings_int8.npy',mmap_mode='r')
    ids=json.loads((OUT/'embeddings_index.json').read_text(encoding='utf-8'))['ids'];lookup={v:i for i,v in enumerate(ids)}
    reports={};maxerror=0.;start=time.perf_counter()
    for task in TASKS:
        print('EVALUAR UNA CABEZA:',task,'CPU batch=1',flush=True)
        head=session(predictor.directory/'heads'/f'{task}.onnx')
        valy=[];valp=[]
        with (ROOT/'data'/f'{task}.jsonl').open(encoding='utf-8') as stream:
            for line in stream:
                row=json.loads(line)
                if row['split']!='validation':continue
                embedding=np.asarray(cache[lookup[row['id_mensaje']]:lookup[row['id_mensaje']]+1])
                valy.append(row[task]);valp.append(float(head.run(None,{'embedding':embedding})[0][0,0]))
        valy=np.asarray(valy);valp=np.asarray(valp)
        # INT8 dinámico puede cambiar con el batch: elegir cortes de producción solo en validation batch 1.
        if task=='sentimiento':
            choices=[(float(lo),float(hi)) for lo in np.arange(.15,.51,.025) for hi in np.arange(.5,.86,.025) if lo<hi]
            cut=max(choices,key=lambda c:metrics(task,valy,valp,c)['f1_macro'])
        else:cut=float(max(np.arange(.01,1.,.01),key=lambda c:metrics(task,valy,valp,c)['f1']))
        ys=[];ps=[];truncated=0;too_long=0
        with (ROOT/'data'/f'{task}.jsonl').open(encoding='utf-8') as stream:
            for line in stream:
                row=json.loads(line)
                if row['split']!='test':continue
                # La evaluación usa exactamente el mismo contrato del servicio.
                try:predictor.validate({'id_mensaje':row['id_mensaje'],'mensaje':row['mensaje']})
                except ValueError:too_long+=1;continue
                embedding=predictor.encode(row['mensaje'])
                expected=cache[lookup[row['id_mensaje']]:lookup[row['id_mensaje']]+1]
                error=float(np.max(np.abs(embedding-expected)));maxerror=max(error,maxerror)
                assert error<1e-5,error
                truncated+=int(bool(predictor.tokenizer.encode(row['mensaje']).overflowing))
                ys.append(row[task]);ps.append(float(head.run(None,{'embedding':embedding})[0][0,0]))
                if len(ys)%200==0:print('test procesados:',len(ys),'RAM MB:',predictor._check_ram(),flush=True)
        reports[task]=dict(threshold=cut,validation_batch1=metrics(task,valy,valp,cut),test_batch1=metrics(task,np.asarray(ys),np.asarray(ps),cut),
                           truncated_test=truncated,rejected_test=too_long)
        predictor.config['thresholds'][task]=cut
        del head;gc.collect()
        print('GUARDADO resultado y sesión liberada:',task,reports[task],flush=True)
    (predictor.directory/'config.json').write_text(json.dumps(predictor.conf

Máximo error embedding caché / paquete: 0.0
Alcance: encoder INT8 y cabeza INT8 reales CPU, test batch 1; umbrales solo validation batch 1
TAREA: es_faq cortes validation batch 1: 0.060000000000000005
VALIDATION:


{'f1': 0.9873150105708245,
 'f1_macro': 0.9877403455220987,
 'f1_per_class': [0.9881656804733728, 0.9873150105708245],
 'confusion_matrix': [[501, 11], [1, 467]],
 'support': 980,
 'positives': 468}

TEST:


...

{'f1_macro': 0.643248578587368,
 'mae': 0.09234553197165951,
 'f1_per_class': [0.2716049382716049, 0.8978102189781022, 0.7603305785123967],
 'confusion_matrix': [[11, 27, 0], [30, 369, 20], [2, 7, 46]],
 'support': 512}

TEST:


{'f1_macro': 0.6577803214783923,
 'mae': 0.09124926367159665,
 'f1_per_class': [0.39285714285714285, 0.9072164948453608, 0.6732673267326733],
 'confusion_matrix': [[11, 12, 0], [20, 308, 18], [2, 13, 34]],
 'support': 418}

Truncados test: 25 rechazados test: 0


0

## 8. Inferencia paso a paso y RAM desde el arranque
El wrapper valida entrada, tokeniza, llama a un solo encoder, obtiene `(1,384)`, carga una cabeza, ejecuta su MLP, guarda el float y libera esa sesión antes de cargar otra. Las solicitudes se serializan con un lock. Se rechaza exceso de caracteres y no se admiten lotes externos.

El supervisor muestrea RSS cada 10 ms desde antes de importar ONNX. Prueba 104 solicitudes, incluidas 8,000 caracteres y truncación a 128 tokens; verifica entradas inválidas. Si observa 1,000 MB, termina el hijo. Se busca además estar bajo 700 MB para dejar margen al SO.

Esta medida corresponde a Windows/Ryzen. La VM OCI completa incluye SO/API/otros servicios y todavía no se ha probado allí. En Linux se incluye un ejemplo de `MemoryMax=700000000` para aplicar un límite al proceso mediante cgroup; hay que integrarlo con el servidor real y repetir la medida. Un worker, batch 1, 128 tokens, 8,000 caracteres.


In [20]:
show_source('serve_mlp_int8.py')
show_source('probe_mlp_memory.py')

IMPLEMENTACIÓN: serve_mlp_int8.py


"""Inferencia CPU con un encoder y UNA cabeza en cada paso. Sin Torch/pandas/sklearn."""
import os
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
import sys,json,threading,time,gc
from pathlib import Path
ROOT=Path(__file__).resolve().parent
sys.path.insert(0,str(ROOT/'deps'))
TASKS=('es_faq','es_logro','sentimiento')

def session(path):
    import onnxruntime as ort
    opts=ort.SessionOptions()
    opts.intra_op_num_threads=opts.inter_op_num_threads=1
    opts.execution_mode=ort.ExecutionMode.ORT_SEQUENTIAL
    opts.graph_optimization_level=ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
    opts.enable_cpu_mem_arena=False;opts.enable_mem_pattern=False
    return ort.InferenceSession(str(path),sess_options=opts,providers=['CPUExecutionProvider'])

class Predictor:
    def __init__(self,deploy_dir=None):
        from tokenizers import Tokenizer
        self.directory=Path(deploy_dir) if deploy_dir else ROOT/'deploy_mlp_int8'
        self.config=json.loads((self.directory/'config.json').read_text(encoding='utf-8'))
        self.lock=threading.Lock()
        self._check_ram()
        self.encoder=session(self.directory/'encoder.onnx')
        self.inputs={i.name for i in self.encoder.get_inputs()}
        self.tokenizer=Tokenizer.from_file(str(self.directory/'tokenizer.json'))
        self.tokenizer.enable_truncation(max_length=self.config['max_tokens'])
        self.tokenizer.no_padding()
        self._check_ram()

    def _check_ram(self):
        import psutil
        value=psutil.Process().memory_info().rss
        if value>=self.config['process_budget_bytes']:raise MemoryError('RSS del proceso alcanzó 1,000 MB')
        return value/1e6

    def validate(self,item):
        import numpy as np
        if not isinstance(item,dict) or set(item)!={'id_mensaje','mensaje'}:raise ValueError('Campos exactos: id_mensaje, mensaje')
        ident=item['id_mensaje']
        if isinstance(ident,bool) or not isinstance(ident,(int,np.integer)) or not -(2**63)<=int(ident)<2**63:raise ValueError('id_mensaje fuera de int64')
        text=item['mensaje']
        if not isinstance(text,str) or not text.strip():raise ValueError('mensaje debe ser texto no vacío')
        if len(text)>self.config['max_chars']:raise ValueError('mensaje supera 8,000 caracteres')
        return int(ident),text

    def encode(self,text,trace=False):
        import numpy as np
        tokens=self.tokenizer.encode(text)
        feed={'input_ids':np.array([tokens.ids],dtype=np.int64),'attention_mask':np.ones((1,len(tokens.ids)),dtype=np.int64),
              'token_type_ids':np.array([tokens.type_ids],dtype=np.int64)}
        if trace:print('[INFERENCIA 1] caracteres:',len(text),'tokens procesados:',len(tokens.ids),'truncado:',bool(tokens.overflowing),'batch: 1',flush=True)
        if trace:print('[INFERENCIA 2] llamada encoder INT8 CPU:',{k:list(v.shape) for k,v in feed.items()},flush=True)
        embedding=self.encoder.run(None,{k:v for k,v in feed.items() if k in self.inputs})[0]
        assert embedding.shape==(1,384)
        if trace:print('[INFERENCIA 3] embedding:',embedding.shape,'primeros 8:',embedding[0,:8].tolist(),'RAM MB:',self._check_ram(),flush=True)
        return embedding

    def __call__(self,item,trace=False):
        ident,text=self.validate(item)
        with self.lock:
            self._check_ram();embedding=self.encode(text,trace);result={'id_mensaje':ident}
            for task in TASKS:
                if trace:print('[INFERENCIA 4] cargar desde SSD:',task,self.config['dims'],'RAM MB:',self._check_ram(),flush=True)
                head=session(self.directory/'heads'/f'{task}.onnx')
                try:
                    value=float(head.run(None,{'embedding':embedding})[0][0,0])
                    if not 0<=value<=1:raise RuntimeError('Score fuera de rango')
                    result[task]=value
                finally:del head
                if trace:print('[INFERENCIA 5]',task,'sigmoi

IMPLEMENTACIÓN: probe_mlp_memory.py


"""Supervisor del proceso de inferencia: pico desde arranque, corte a 1,000 MB."""
import sys,json,subprocess,time
from pathlib import Path
import psutil
ROOT=Path(__file__).resolve().parent
OUT=ROOT/'artifacts_mlp_int8'

def main():
    report=OUT/'memory.json';log=OUT/'inference.log';peak=0;samples=0;exceeded=False
    command=[sys.executable,'-u',str(ROOT/'serve_mlp_int8.py'),'--report',str(report)]
    print('LLAMADA:',command,flush=True)
    with log.open('w',encoding='utf-8') as stream:
        child=subprocess.Popen(command,stdout=stream,stderr=subprocess.STDOUT)
        process=psutil.Process(child.pid)
        while child.poll() is None:
            try:peak=max(peak,process.memory_info().rss);samples+=1
            except psutil.NoSuchProcess:break
            if peak>=1_000_000_000:exceeded=True;child.terminate();break
            time.sleep(.01)
        code=child.wait()
    print(log.read_text(encoding='utf-8'),flush=True)
    if exceeded:raise MemoryError('Supervisor detuvo inferencia: alcanzó 1,000 MB RSS')
    if code:raise RuntimeError(f'Inferencia falló: {code}')
    result=json.loads(report.read_text(encoding='utf-8'))
    result.update(cold_to_hot_peak_rss_mb=peak/1e6,sampling_ms=10,samples=samples,budget_bytes=1_000_000_000,
        under_1000_mb=peak<1_000_000_000,under_700_mb=peak<700_000_000,scope='Proceso Windows local; no incluye SO/API ni comprueba OCI')
    assert not result['imported_heavy_modules']
    assert result['under_1000_mb']
    report.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    print('PICO DE RAM DESDE ARRANQUE MB:',peak/1e6,'<1000:',result['under_1000_mb'],'<700:',result['under_700_mb'],flush=True)

if __name__=='__main__':
    sys.stdout.reconfigure(encoding='utf-8');main()

In [21]:
execute_step([ROOT/'probe_mlp_memory.py'])
memory=json.loads((OUT/'memory.json').read_text(encoding='utf-8'))
print('PICO desde arranque MB:',memory['cold_to_hot_peak_rss_mb'])
print('¿Debajo de 1,000 MB?',memory['under_1000_mb'],'¿Debajo de 700 MB?',memory['under_700_mb'])
print('RAM por etapa MB:',memory['stages_mb'])
print('Latencia p50 / p95 ms:',memory['p50_ms'],memory['p95_ms'])
print('Checks de rechazo:',memory['boundary_checks'],'imports de librerías pesadas:',memory['imported_heavy_modules'])
print('SALIDA REAL del wrapper:');display(memory['example'])
assert memory['under_1000_mb'] and not memory['imported_heavy_modules']
del memory;gc.collect()


LLAMADA EXPLÍCITA: ['C:\\Python311\\python.exe', '-u', 'C:\\Users\\saul_\\Documents\\Codex\\2026-10-04\\github_delivery\\data_science\\2026-10-04\\probe_mlp_memory.py']


LLAMADA: ['C:\\Python311\\python.exe', '-u', 'C:\\Users\\saul_\\Documents\\Codex\\2026-10-04\\github_delivery\\data_science\\2026-10-04\\serve_mlp_int8.py', '--report', 'C:\\Users\\saul_\\Documents\\Codex\\2026-10-04\\github_delivery\\data_science\\2026-10-04\\artifacts_mlp_int8\\memory.json']


[INFERENCIA 1] caracteres: 39 tokens procesados: 11 truncado: False batch: 1


[INFERENCIA 2] llamada encoder INT8 CPU: {'input_ids': [1, 11], 'attention_mask': [1, 11], 'token_type_ids': [1, 11]}


[INFERENCIA 3] embedding: (1, 384) primeros 8: [0.2182409167289734, -0.03402449190616608, 0.3634309470653534, -0.3045717775821686, 0.00033970398362725973, 0.13664472103118896, 0.512833833694458, 0.9588077068328857] RAM MB: 347.987968


...

PICO DE RAM DESDE ARRANQUE MB: 357.134336 <1000: True <700: True


Proceso terminado; sus modelos y memoria quedaron liberados.


PICO desde arranque MB: 357.134336
¿Debajo de 1,000 MB? True ¿Debajo de 700 MB? True
RAM por etapa MB: {'before_load': 30.851072, 'after_load': 345.976832, 'after_inference': 353.697792}
Latencia p50 / p95 ms: 8.256500004790723 30.71431999487686
Checks de rechazo: 5 imports de librerías pesadas: []
SALIDA REAL del wrapper:


{'id_mensaje': 123,
 'es_faq': 0.00041484832763671875,
 'es_logro': 0.5956413745880127,
 'sentimiento': 0.9855363368988037}

0

## 9. Contrato y uso del wrapper
Entrada exacta: `{id_mensaje: int64, mensaje: string}`. ID se conserva; máximo 8,000 caracteres Unicode, texto no vacío, primeros 128 tokens. Salida exacta: `{id_mensaje: int64, es_faq: float[0,1], es_logro: float[0,1], sentimiento: float[0,1]}`.

`serve_mlp_int8.Predictor()` se instancia una vez por proceso. `predictor(entrada, trace=True)` muestra pasos. En OCI no se importa runtime.py ni los scripts de entrenamiento. `serve_jsonl.py` ofrece entrada/salida JSON por línea con el mismo predictor persistente. Los errores de validación se devuelven aparte de las respuestas exitosas.

Instalar dependencias CPU Linux usando requirements-serving.txt; los wheels/DLL de Windows en deps/ no se suben ni se copian a OCI. Conservar encoder.onnx, weights-*.bin, tokenizer.json, heads/ y config.json juntos.


In [22]:
show_source('serve_jsonl.py')
print((ROOT/'mlp-oci.service.example').read_text(encoding='utf-8'))

IMPLEMENTACIÓN: serve_jsonl.py


"""Un proceso, un encoder: JSON por línea en stdin; respuesta JSON por línea en stdout."""
import sys,json
from serve_mlp_int8 import Predictor

def main():
    predictor=Predictor()
    print('Encoder INT8 listo; batch=1, 128 tokens, 8000 caracteres, un worker.',file=sys.stderr,flush=True)
    # 8000 caracteres UTF-8 usan como máximo 32000 bytes; margen para envoltorio JSON.
    # Rechazar un sobre grande ANTES de cargarlo completo en memoria.
    max_line=40000
    while True:
        line=sys.stdin.buffer.readline(max_line+1)
        if not line:break
        try:
            if len(line)>max_line:
                while not line.endswith(b'\n'):
                    line=sys.stdin.buffer.readline(max_line+1)
                    if not line:break
                raise ValueError('Sobre JSON supera 40000 bytes')
            item=json.loads(line.decode('utf-8'))
            result=predictor(item)
        except (ValueError,TypeError,UnicodeError) as error:
            result={'error':str(error)}
        print(json.dumps(result,ensure_ascii=False,allow_nan=False),flush=True)

if __name__=='__main__':
    sys.stdout.reconfigure(encoding='utf-8');main()

# Ejemplo de integración en Linux; adaptar usuario y ruta. No se ha desplegado en OCI.
# El padre conecta JSONL mediante un socket/pipe: stdin de este ejemplo no es una API HTTP.
[Service]
Type=simple
WorkingDirectory=/opt/mlp/2026-10-04
ExecStart=/opt/mlp/.venv/bin/python /opt/mlp/2026-10-04/serve_jsonl.py
Environment=OMP_NUM_THREADS=1
Environment=TOKENIZERS_PARALLELISM=false
MemoryHigh=600000000
MemoryMax=700000000
TasksMax=32
Restart=no



## 10. Registros anteriores y conclusión
La comparación original `384→1` frente a `384→16→4→1` se conserva en `artifacts_v2/comparison.json` y en el notebook anterior. Usó el afinado del encoder y una política de ventanas distinta; no debe mezclarse con esta comparación controlada sobre el mismo encoder INT8 congelado. PCA fue un experimento anterior y no forma parte del paquete final.

La profunda cumple el diseño pedido y cuesta poco; no ofrece una mejora de F1 demostrada sobre la pequeña. El encoder INT8 y un runtime CPU liviano son los cambios principales para RAM. FAQ funciona bien en estos datos; logro y sentimiento todavía tienen errores visibles y requieren validación con mensajes del dominio real. No se afirma que el proceso completo en OCI ya esté validado.

Fuentes técnicas: [cuantización ONNX Runtime](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html), [pesos externos ONNX](https://onnx.ai/onnx/api/external_data_helper.html), [shapes OCI](https://docs.oracle.com/en-us/iaas/Content/Compute/References/computeshapes.htm).
